# Чистый ноутбук для fine-tuning LEGATO

Этот ноутбук рассчитан на уже подготовленное совместимое окружение и не меняет файлы проекта.

Что делает:
- читает датасет с колонками `image_path` и `abc_path` или уже готовыми `image`/`transcription`
- анализирует словарь ABC
- добавляет маркеры `<TITLE>`, `<LYRICS>`, `<ANNOT>`
- загружает чекпоинт LEGATO
- замораживает почти всю модель для теста
- обучает с взвешенным loss


In [2]:
from __future__ import annotations

import json
import os
import string
import sys
import unicodedata
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable
import re

import numpy as np
import torch
import torch.distributed as dist
from datasets import Dataset, DatasetDict, load_from_disk
from huggingface_hub import login, snapshot_download
from PIL import Image
from safetensors.torch import load_file as load_safetensors_file
from torch import nn
from transformers import AutoConfig, AutoModel, AutoProcessor, Seq2SeqTrainingArguments, TrainerCallback, set_seed
from transformers import MllamaVisionModel


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'legato').exists() and (candidate / 'scripts' / 'train.py').exists():
            return candidate
    raise FileNotFoundError('Не удалось найти корень проекта LEGATO.')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from legato.metrics import compute_error_rates
from legato.models import LegatoModel, LegatoConfig
from legato.trainer import LegatoTrainer

HF_TOKEN = os.environ.get('HF_TOKEN')
if HF_TOKEN:
    login(token=HF_TOKEN)
    print('HF token загружен из окружения.')
else:
    print('HF_TOKEN не найден. Если вы уже залогинены через huggingface_hub, это не страшно.')

NOTEBOOK_ROOT = PROJECT_ROOT / 'legato-utils-new'
ARTIFACT_ROOT = NOTEBOOK_ROOT / 'artifacts'
OUTPUT_ROOT = NOTEBOOK_ROOT / 'outputs'
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('NOTEBOOK_ROOT =', NOTEBOOK_ROOT)


def load_repo_state_dict(repo_id: str, token: str | None = None) -> dict[str, torch.Tensor]:
    snapshot_path = Path(snapshot_download(repo_id=repo_id, token=token))
    safetensors_files = sorted(snapshot_path.glob('*.safetensors'))
    if not safetensors_files:
        raise FileNotFoundError(f'В snapshot не найдено *.safetensors: {snapshot_path}')
    merged = {}
    for safetensors_path in safetensors_files:
        merged.update(load_safetensors_file(str(safetensors_path)))
    return merged


def remap_legato_keys_for_current_mllama(state_dict: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
    remapped = {}
    for key, value in state_dict.items():
        new_key = key
        if key.startswith('language_model.model.'):
            new_key = 'model.language_model.' + key[len('language_model.model.'):]
        elif key.startswith('language_model.lm_head.'):
            new_key = 'lm_head.' + key[len('language_model.lm_head.'):]
        elif key.startswith('language_model.'):
            new_key = 'model.' + key
        elif key.startswith('multi_modal_projector.'):
            new_key = 'model.multi_modal_projector.' + key[len('multi_modal_projector.'):]
        elif key.startswith('vision_model.'):
            # Не грузим vision из LEGATO-чекпоинта: ниже загрузим encoder отдельно из base vision model.
            continue
        remapped[new_key] = value
    return remapped


def load_legato_model_clean(model_ref: str, token: str | None = None):
    config = AutoConfig.from_pretrained(model_ref, token=token)
    assert isinstance(config, LegatoConfig), f'Ожидался LegatoConfig, получили: {type(config)}'
    model = LegatoModel(config, load_pretrained_encoder=False)

    raw_state_dict = load_repo_state_dict(model_ref, token=token)
    remapped_state_dict = remap_legato_keys_for_current_mllama(raw_state_dict)
    incompatible = model.load_state_dict(remapped_state_dict, strict=False)

    encoder_ref = getattr(config, 'encoder_pretrained_model_name_or_path', None)
    if encoder_ref is None:
        raise ValueError('В конфиге отсутствует encoder_pretrained_model_name_or_path')
    model.model.vision_model = MllamaVisionModel.from_pretrained(encoder_ref, token=token)
    model.config.vision_config = model.model.vision_model.config
    for param in model.model.vision_model.parameters():
        param.requires_grad = False

    print('missing keys:', len(incompatible.missing_keys))
    print('unexpected keys:', len(incompatible.unexpected_keys))
    if incompatible.missing_keys:
        print('Первые missing keys:', incompatible.missing_keys[:20])
    if incompatible.unexpected_keys:
        print('Первые unexpected keys:', incompatible.unexpected_keys[:20])

    return model


HF_TOKEN не найден. Если вы уже залогинены через huggingface_hub, это не страшно.
PROJECT_ROOT = G:\Diplom\legato
NOTEBOOK_ROOT = G:\Diplom\legato\legato-utils-new


In [3]:
@dataclass
class PipelineConfig:
    base_model: str = 'guangyangmusic/legato'
    dataset_path: str = r'G:\Diplom\datasets\PDMX\lyrics_dataset_v5\hf_dataset'
    output_name: str = 'legato-text-music-clean-tiny'
    seed: int = 42
    abc_encoding: str = 'utf-8'
    markers: tuple[str, ...] = tuple()
    force_add_characters: tuple[str, ...] = ('&', ';', '@', 'J', 'W', 'Y', 'Z', 'j', '\xa0', '´')
    music_loss_weight: float = 1.0
    train_split_name: str = 'train'
    val_split_name: str = 'validation'
    test_split_name: str = 'test'
    max_train_samples: int = 20000
    max_val_samples: int = 200
    max_test_samples: int = 150
    num_train_epochs: int = 4
    max_steps: int = -1
    learning_rate: float = 1e-5
    per_device_train_batch_size: int = 1
    per_device_eval_batch_size: int = 1
    gradient_accumulation_steps: int = 4
    dataloader_num_workers: int = 0
    generation_max_length: int = 512
    generation_num_beams: int = 1
    logging_steps: int = 100
    eval_steps: int = 4658
    save_steps: int = 4658
    train_last_n_decoder_layers: int = 6
    train_lm_head: bool = True
    train_token_embeddings: bool = True
    run_train: bool = True
    run_eval: bool = True


CFG = PipelineConfig()
set_seed(CFG.seed)

DATASET_PATH = CFG.dataset_path
RUN_ROOT = OUTPUT_ROOT / CFG.output_name
PROCESSOR_OUT = RUN_ROOT / 'processor_extended'
MODEL_OUT = RUN_ROOT / 'model'
METRICS_LOG_PATH = RUN_ROOT / 'metrics_log.txt'
REPORT_PATH = ARTIFACT_ROOT / f'{CFG.output_name}_vocab_report.json'
META_PATH = ARTIFACT_ROOT / f'{CFG.output_name}_tokenizer_meta.json'

RUN_ROOT.mkdir(parents=True, exist_ok=True)
PROCESSOR_OUT.mkdir(parents=True, exist_ok=True)
MODEL_OUT.mkdir(parents=True, exist_ok=True)
METRICS_LOG_PATH.write_text('', encoding='utf-8')

print(CFG)


PipelineConfig(base_model='guangyangmusic/legato', dataset_path='G:\\Diplom\\datasets\\PDMX\\lyrics_dataset_v5\\hf_dataset', output_name='legato-text-music-clean-tiny', seed=42, abc_encoding='utf-8', markers=(), force_add_characters=('&', ';', '@', 'J', 'W', 'Y', 'Z', 'j', '\xa0', '´'), music_loss_weight=1.0, train_split_name='train', val_split_name='validation', test_split_name='test', max_train_samples=20000, max_val_samples=200, max_test_samples=150, num_train_epochs=4, max_steps=-1, learning_rate=1e-05, per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=4, dataloader_num_workers=0, generation_max_length=512, generation_num_beams=1, logging_steps=100, eval_steps=4658, save_steps=4658, train_last_n_decoder_layers=6, train_lm_head=True, train_token_embeddings=True, run_train=True, run_eval=True)


## 1. Нормализация датасета

In [4]:
def resolve_existing_path(raw_path: str | Path, base_dir: Path) -> Path:
    candidate = Path(raw_path)
    if candidate.is_absolute() and candidate.exists():
        return candidate
    variants = [base_dir / candidate, PROJECT_ROOT / candidate, candidate]
    for path in variants:
        if path.exists():
            return path.resolve()
    raise FileNotFoundError(f'Не найден путь: {raw_path}')


NAME_ATTR_RE = re.compile(r'\b(?:nm|snm)="([^"]+)"')


def build_transcription_from_abc(abc_text: str) -> str:
    abc_text = abc_text.replace('\r', '\n').replace('\xa0', ' ')
    cleaned_lines = []

    for raw_line in abc_text.splitlines():
        line = raw_line.strip()
        if not line:
            continue

        if line.startswith('w:'):
            continue

        if line.startswith('V:'):
            line = re.sub(r'\s+nm="[^"]*"', '', line)
            line = re.sub(r'\s+snm="[^"]*"', '', line)
            line = re.sub(r'\s+', ' ', line).strip()

        cleaned_lines.append(line)

    return '\n'.join(cleaned_lines)


def load_transcription_from_split(split_ds):
    if 'transcription' in split_ds.column_names:
        return [str(x) if x is not None else '' for x in split_ds['transcription']]

    if 'abc' in split_ds.column_names:
        return [build_transcription_from_abc(str(x)) if x is not None else '' for x in split_ds['abc']]

    for key in ['abc_path', 'transcription_path', 'label_path', 'abc_file', 'annotation_path']:
        if key in split_ds.column_names:
            paths = split_ds[key]
            outputs = []
            for raw_path in paths:
                if raw_path is None:
                    outputs.append('')
                    continue
                abc_file = resolve_existing_path(raw_path, DATASET_PATH)
                outputs.append(build_transcription_from_abc(abc_file.read_text(encoding=CFG.abc_encoding)))
            return outputs

    raise KeyError(f"?? ??????? ???? ? ABC/?????????????. ????????? ????: {split_ds.column_names}")


def normalize_split(split_ds, split_name: str):
    transcriptions = load_transcription_from_split(split_ds)
    normalized = split_ds
    if 'transcription' in normalized.column_names:
        normalized = normalized.remove_columns(['transcription'])
    if 'split_name' in normalized.column_names:
        normalized = normalized.remove_columns(['split_name'])
    normalized = normalized.add_column('transcription', transcriptions)
    normalized = normalized.add_column('split_name', [split_name] * len(normalized))
    return normalized


raw_dataset = load_from_disk(str(DATASET_PATH))
assert isinstance(raw_dataset, DatasetDict), 'Ожидался DatasetDict с train/validation/test.'

print('Найденные сплиты:', list(raw_dataset.keys()))

train_raw = raw_dataset[CFG.train_split_name].select(range(min(CFG.max_train_samples, len(raw_dataset[CFG.train_split_name]))))
val_raw = raw_dataset[CFG.val_split_name].select(range(min(CFG.max_val_samples, len(raw_dataset[CFG.val_split_name]))))
test_raw = raw_dataset[CFG.test_split_name].select(range(min(CFG.max_test_samples, len(raw_dataset[CFG.test_split_name]))))

dataset = DatasetDict({
    'train': normalize_split(train_raw, 'train'),
    'val': normalize_split(val_raw, 'val'),
    'test': normalize_split(test_raw, 'test'),
})

for split_name, split_ds in dataset.items():
    print(split_name, len(split_ds), split_ds.column_names)


Найденные сплиты: ['train', 'validation', 'test']
train 18632 ['id', 'image', 'abc_path', 'image_path', 'mxl_path', 'title', 'composer_name', 'n_lyrics', 'abc_chars', 'target_tokens', 'lyrics_lines', 'transcription', 'split_name']
val 200 ['id', 'image', 'abc_path', 'image_path', 'mxl_path', 'title', 'composer_name', 'n_lyrics', 'abc_chars', 'target_tokens', 'lyrics_lines', 'transcription', 'split_name']
test 100 ['id', 'image', 'abc_path', 'image_path', 'mxl_path', 'title', 'composer_name', 'n_lyrics', 'abc_chars', 'target_tokens', 'lyrics_lines', 'transcription', 'split_name']


In [5]:
print(dataset['train'][0]['transcription'][:4000])
print(dataset['val'][1]['transcription'][:4000])

X:1
T:De Colores [C]
L:1/8
M:6/8
I:linebreak $
K:C
V:1 treble
V:1
 z2 |"C" [EG]3-"G7" [EG]2 [DF] |"C" [DF] [CE]2 z [CE][DF] | [EG][EG]>[EG] [EG][DF][CE] | %4
w:|De * co-|lo- res, de co-|lo- res se vis- ten los|
 [EG][EG]>[EG] [EG][DF][CE] |"G7" [EG] [DF]2- [DF]2 z |"G7" [DF]3- [DF]2 [CE] |$ %7
w:cam- pos en la pri- ma-|ve- ra *|De * co-|
 [CE] [B,D]2 z [DF][EG] | [FA][FA]>[FA] [FA][EG][DF] | [FA][FA]>[FA] [FA][EG][DF] | %10
w:lo- res, de co-|lo- res son los pa- ja-|ri- tos que vie- nen de~a-|
"C" [FA] [EG]2- [EG]2 z |"C" [EG]3-"G7" [EG]2 [DF] |"C" [DF] [CE]2 z [CE][DF] |$ %13
w:fue- ra. *|De * co-|lo- res, de co-|
 [EG][EG][EG] [EG][DF][CE] | [EG][EG]>[EG] [EG][FA][G_B] |"F" [FA]3 z |: [DF][EG] | %17
w:lo- res es el ar- co|i- ris que ve- mos lu-|cir.|Y por|
"F" [FA][FA][FA]"G7" [FA][GB]>[FA] |"C" [FA][EG]>[EG]"Am" [EG][FA][EG] |33$ %19
w:e- so los gran- des a-|mo- res de mu- chos co-|
"Dm" [EG][DF][EG]"G7" [FA][EG][DF] |"C" [CE]3 z :|33"Dm" [EG][DF][EG]"G7" [FA][EG][DFB] | %22
w:lo- re

## 2. Анализ словаря и расширение tokenizer

In [6]:
base_processor = AutoProcessor.from_pretrained(CFG.base_model, token=HF_TOKEN)
base_tokenizer = base_processor.tokenizer


def iter_transcriptions(ds_dict: DatasetDict) -> Iterable[str]:
    for split_ds in ds_dict.values():
        for item in split_ds['transcription']:
            if item is not None:
                yield item


def contains_unknown_token(tokenizer, text: str) -> bool:
    ids = tokenizer(text, add_special_tokens=False)['input_ids']
    if not ids:
        return True
    tokens = tokenizer.convert_ids_to_tokens(ids)
    unk_token = getattr(tokenizer, 'unk_token', None)
    if unk_token and unk_token in tokens:
        return True
    normalized_tokens = [str(tok).lower() for tok in tokens]
    return any('unk' in tok for tok in normalized_tokens)


char_counter = Counter()
num_transcriptions_seen = 0
for item in iter_transcriptions(dataset):
    char_counter.update(item)
    num_transcriptions_seen += 1
unique_chars = sorted(char_counter)
forced_chars = sorted(set(CFG.force_add_characters))

dataset_chars_with_unk = sorted(ch for ch in unique_chars if contains_unknown_token(base_tokenizer, ch))
forced_chars_with_unk = sorted(ch for ch in forced_chars if contains_unknown_token(base_tokenizer, ch))
all_chars_with_unk = sorted(set(dataset_chars_with_unk) | set(forced_chars_with_unk))
chars_to_add = sorted(ch for ch in forced_chars_with_unk)
excluded_chars_with_unk = sorted(ch for ch in all_chars_with_unk if ch not in chars_to_add)

report = {
    'base_model': CFG.base_model,
    'dataset_path': str(DATASET_PATH),
    'num_train_items': len(dataset['train']),
    'num_val_items': len(dataset['val']),
    'num_test_items': len(dataset['test']),
    'num_transcriptions_seen': num_transcriptions_seen,
    'markers': list(CFG.markers),
    'chars_to_add': chars_to_add,
    'dataset_chars_with_unk': dataset_chars_with_unk,
    'forced_chars_with_unk': forced_chars_with_unk,
    'excluded_chars_with_unk': excluded_chars_with_unk,
    'top_100_char_counts': dict(char_counter.most_common(100)),
}
REPORT_PATH.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')

processor = AutoProcessor.from_pretrained(CFG.base_model, token=HF_TOKEN)
tokenizer = processor.tokenizer
original_vocab_size = len(tokenizer)

existing_vocab = tokenizer.get_vocab()
marker_tokens_to_add = [tok for tok in CFG.markers if tok not in existing_vocab]
char_tokens_to_add = [tok for tok in chars_to_add if tok not in existing_vocab]

num_added_markers = tokenizer.add_special_tokens({'additional_special_tokens': marker_tokens_to_add}) if marker_tokens_to_add else 0
num_added_chars = tokenizer.add_tokens(char_tokens_to_add) if char_tokens_to_add else 0
processor.save_pretrained(PROCESSOR_OUT)

tokenizer_meta = {
    'base_model': CFG.base_model,
    'processor_out': str(PROCESSOR_OUT),
    'original_vocab_size': original_vocab_size,
    'extended_vocab_size': len(tokenizer),
    'num_added_markers': num_added_markers,
    'num_added_chars': num_added_chars,
    'marker_tokens': list(CFG.markers),
    'marker_token_ids': [tokenizer.convert_tokens_to_ids(tok) for tok in CFG.markers],
    'dataset_chars_with_unk': dataset_chars_with_unk,
    'forced_chars_with_unk': forced_chars_with_unk,
    'excluded_chars_with_unk': excluded_chars_with_unk,
    'char_tokens_added': char_tokens_to_add,
}
META_PATH.write_text(json.dumps(tokenizer_meta, indent=2, ensure_ascii=False), encoding='utf-8')
print('unk_token:', getattr(tokenizer, 'unk_token', None))
print('unk_token_id:', getattr(tokenizer, 'unk_token_id', None))
print('Символы из датасета с unk:', dataset_chars_with_unk)
print('Принудительно проверенные символы с unk:', forced_chars_with_unk)
print('Исключенные символы с unk:', excluded_chars_with_unk)
print('Символы, выбранные для добавления:', char_tokens_to_add)
print(json.dumps(tokenizer_meta, indent=2, ensure_ascii=False))


unk_token: None
unk_token_id: None
Символы из датасета с unk: ['&', ';', '@', 'J', 'W', 'Y', 'Z', '`', 'j']
Принудительно проверенные символы с unk: ['&', ';', '@', 'J', 'W', 'Y', 'Z', 'j', '\xa0', '´']
Исключенные символы с unk: ['`']
Символы, выбранные для добавления: ['&', ';', '@', 'J', 'W', 'Y', 'Z', 'j', '\xa0', '´']
{
  "base_model": "guangyangmusic/legato",
  "processor_out": "G:\\Diplom\\legato\\legato-utils-new\\outputs\\legato-text-music-clean-tiny\\processor_extended",
  "original_vocab_size": 4097,
  "extended_vocab_size": 4107,
  "num_added_markers": 0,
  "num_added_chars": 10,
  "marker_tokens": [],
  "marker_token_ids": [],
  "dataset_chars_with_unk": [
    "&",
    ";",
    "@",
    "J",
    "W",
    "Y",
    "Z",
    "`",
    "j"
  ],
  "forced_chars_with_unk": [
    "&",
    ";",
    "@",
    "J",
    "W",
    "Y",
    "Z",
    "j",
    " ",
    "´"
  ],
  "excluded_chars_with_unk": [
    "`"
  ],
  "char_tokens_added": [
    "&",
    ";",
    "@",
    "J",
    "W",


## 3. Взвешенный trainer

In [7]:
def build_token_weight_vector(vocab_size: int, downweighted_token_ids: list[int], default_weight: float, downweight: float) -> torch.Tensor:
    weights = torch.full((vocab_size,), float(default_weight), dtype=torch.float32)
    if downweighted_token_ids:
        weights[torch.tensor(downweighted_token_ids, dtype=torch.long)] = float(downweight)
    return weights


MUSIC_TOKEN_RE = re.compile(r'^[\[\]\(\)\{\}\|:\d\/,\.\-_=\^~!%$+*?#&\"<>A-Ga-gzZxX]+$')


def is_music_like_token(token: str) -> bool:
    token = str(token)
    if not token:
        return False
    stripped = token.strip()
    if not stripped:
        return False
    if stripped in {'A', 'B', 'C', 'D', 'E', 'F', 'G', 'a', 'b', 'c', 'd', 'e', 'f', 'g'}:
        return False
    lowered = stripped.lower()
    if lowered.startswith(('t:', 'c:', 'w:', 'x:', 'l:', 'q:', 'm:', 'i:', 'k:', 'v:', 'r:', 's:', 'n:', 'z:', '%%score')):
        return False
    return MUSIC_TOKEN_RE.fullmatch(stripped) is not None


special_ids = set(tokenizer.all_special_ids)
downweighted_token_ids = []
downweighted_token_texts = []
for idx in range(original_vocab_size):
    if idx in special_ids:
        continue
    token_text = tokenizer.convert_ids_to_tokens(idx)
    if is_music_like_token(token_text):
        downweighted_token_ids.append(idx)
        downweighted_token_texts.append(token_text)

token_weight_vector = build_token_weight_vector(
    vocab_size=len(tokenizer),
    downweighted_token_ids=downweighted_token_ids,
    default_weight=1.0,
    downweight=CFG.music_loss_weight,
)

print('Music-like downweighted tokens:', len(downweighted_token_ids), 'of', original_vocab_size)
print('First music-like tokens:', downweighted_token_texts[:40])


class WeightedLegatoTrainer(LegatoTrainer):
    def __init__(self, *args, token_weight_vector: torch.Tensor, **kwargs):
        super().__init__(*args, **kwargs)
        self._token_weight_vector = token_weight_vector.clone().float()
        self._token_weight_vector_device = None

    def _get_token_weight_vector(self, device: torch.device) -> torch.Tensor:
        if self._token_weight_vector_device != device:
            self._token_weight_vector = self._token_weight_vector.to(device)
            self._token_weight_vector_device = device
        return self._token_weight_vector

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        inputs = {k: v for k, v in inputs.items() if not k.startswith('gen_')}
        labels = inputs['labels']
        model_inputs = {k: v for k, v in inputs.items() if k != 'labels'}
        outputs = model(**model_inputs, use_cache=False)
        logits = outputs.logits.float()

        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()

        loss_fct = nn.CrossEntropyLoss(reduction='none')
        token_losses = loss_fct(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
        ).view_as(shift_labels)

        weight_vector = self._get_token_weight_vector(shift_logits.device)
        safe_labels = shift_labels.clamp(min=0)
        token_weights = weight_vector[safe_labels]
        token_weights = torch.where(shift_labels.eq(-100), torch.zeros_like(token_weights), token_weights)

        weighted_loss = (token_losses * token_weights).sum() / token_weights.sum().clamp_min(1e-8)
        return (weighted_loss, outputs) if return_outputs else weighted_loss


class TextMetricsLoggerCallback(TrainerCallback):
    def __init__(self, log_path: Path):
        self.log_path = Path(log_path)

    def _append(self, payload: dict):
        with self.log_path.open('a', encoding='utf-8') as f:
            f.write(json.dumps(payload, ensure_ascii=False) + '\n')

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            payload = {'event': 'log', 'step': state.global_step, **logs}
            self._append(payload)

    def on_save(self, args, state, control, **kwargs):
        payload = {
            'event': 'save',
            'step': state.global_step,
            'checkpoint_dir': str(Path(args.output_dir) / f'checkpoint-{state.global_step}'),
        }
        self._append(payload)


Music-like downweighted tokens: 3638 of 4097
First music-like tokens: ['!', '"', '#', '$', '%', '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', '<', '=', '>', '?', 'X', '[', ']', '^', '_', 'x', 'z', '{', '|', '}', '~', ' z']


## 4. Загрузка модели и сильная заморозка

In [8]:
model = load_legato_model_clean(CFG.base_model, token=HF_TOKEN)
model.resize_token_embeddings(len(tokenizer))


def freeze_all_parameters(model):
    for param in model.parameters():
        param.requires_grad = False


def get_decoder_layers(model):
    candidates = [
        ('model.language_model.layers', getattr(getattr(getattr(model, 'model', None), 'language_model', None), 'layers', None)),
        ('model.language_model.model.layers', getattr(getattr(getattr(getattr(model, 'model', None), 'language_model', None), 'model', None), 'layers', None)),
        ('language_model.layers', getattr(getattr(model, 'language_model', None), 'layers', None)),
        ('language_model.model.layers', getattr(getattr(getattr(model, 'language_model', None), 'model', None), 'layers', None)),
    ]
    for name, layers in candidates:
        if layers is not None:
            return name, layers
    raise AttributeError('Не удалось найти слои декодера.')


freeze_all_parameters(model)
for param in model.model.vision_model.parameters():
    param.requires_grad = False

decoder_path, decoder_layers = get_decoder_layers(model)
for layer in list(decoder_layers)[-CFG.train_last_n_decoder_layers:]:
    for param in layer.parameters():
        param.requires_grad = True

if CFG.train_token_embeddings:
    for param in model.get_input_embeddings().parameters():
        param.requires_grad = True

if CFG.train_lm_head:
    for param in model.get_output_embeddings().parameters():
        param.requires_grad = True

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)

print('Путь к слоям декодера:', decoder_path)
print('Обучаемые параметры:', f'{trainable_params:,}')
print('Замороженные параметры:', f'{frozen_params:,}')


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


missing keys: 0
unexpected keys: 0
Путь к слоям декодера: model.language_model.layers
Обучаемые параметры: 38,029,954
Замороженные параметры: 905,631,129


## 5. Trainer и tiny-конфиг

In [9]:
def get_metric_target(examples):
    return {
        'label_ids': tokenizer(
            examples['transcription'],
            add_special_tokens=False,
            truncation=False,
        )['input_ids'],
    }


map_num_proc = CFG.dataloader_num_workers if CFG.dataloader_num_workers > 0 else None

metric_targets = dataset['val'].map(
    get_metric_target,
    remove_columns=dataset['val'].column_names,
    num_proc=map_num_proc,
    batched=True,
).to_dict()
tokens_to_mask = torch.tensor([tokenizer.convert_tokens_to_ids(tok) for tok in CFG.markers] + [tokenizer.pad_token_id])


def collate_fn(examples):
    outputs = processor(
        images=[example['image'].convert('RGB') if hasattr(example['image'], 'convert') else example['image'] for example in examples],
        text=[example['transcription'] for example in examples],
        return_num_tiles=True,
        truncation=True,
        padding='max_length',
        return_tensors='pt',
    )
    gen_outputs = processor(
        num_tiles=outputs.pop('num_tiles'),
        truncation=True,
        padding=True,
        return_tensors='pt',
    )
    outputs.update({
        f'gen_{k}': outputs[k] if k not in gen_outputs else gen_outputs[k]
        for k in outputs
    })
    outputs['labels'] = outputs['input_ids'].clone().masked_fill(torch.isin(outputs['input_ids'], tokens_to_mask), -100)
    return outputs


special_tokens = [tokenizer.bos_token_id, tokenizer.eos_token_id, tokenizer.pad_token_id, -100]


def remove_special_tokens(array):
    masks = np.isin(array, special_tokens, invert=True)
    return [a[mask] for a, mask in zip(array, masks)]


def metric_fn(p):
    preds = remove_special_tokens(p.predictions)
    metric_num_workers = max(1, CFG.dataloader_num_workers)
    results = [
        compute_error_rates(
            tokenizer,
            metric_num_workers,
            *metric_targets.values(),
            preds,
        )
    ] if getattr(training_args, 'process_index', 0) == 0 else [None]

    if dist.is_available() and dist.is_initialized():
        dist.broadcast_object_list(results, src=0)

    return results[0]



training_args = Seq2SeqTrainingArguments(
    output_dir=str(MODEL_OUT),
    remove_unused_columns=False,
    do_train=True,
    do_eval=True,
    predict_with_generate=True,
    metric_for_best_model='eval_SER',
    greater_is_better=False,
    load_best_model_at_end=False,
    num_train_epochs=CFG.num_train_epochs,
    max_steps=CFG.max_steps,
    learning_rate=CFG.learning_rate,
    per_device_train_batch_size=CFG.per_device_train_batch_size,
    per_device_eval_batch_size=CFG.per_device_eval_batch_size,
    gradient_accumulation_steps=CFG.gradient_accumulation_steps,
    dataloader_num_workers=CFG.dataloader_num_workers,
    logging_steps=CFG.logging_steps,
    save_strategy='steps',
    save_total_limit=2,
    save_steps=CFG.save_steps,
    eval_strategy='steps',
    eval_steps=CFG.eval_steps,
    generation_max_length=CFG.generation_max_length,
    generation_num_beams=CFG.generation_num_beams,
    bf16=torch.cuda.is_available(),
    bf16_full_eval=torch.cuda.is_available(),
    report_to='none',
    seed=CFG.seed,
    logging_dir=str(MODEL_OUT / 'logs'),
)

trainer = WeightedLegatoTrainer(
    model=model,
    args=training_args,
    token_weight_vector=token_weight_vector,
    data_collator=collate_fn,
    train_dataset=dataset['train'],
    eval_dataset=dataset['val'],
    compute_metrics=metric_fn,
)

trainer.add_callback(TextMetricsLoggerCallback(METRICS_LOG_PATH))

print('Trainer готов.')
print('Metrics log:', METRICS_LOG_PATH)


Trainer готов.
Metrics log: G:\Diplom\legato\legato-utils-new\outputs\legato-text-music-clean-tiny\metrics_log.txt


## 6. Запуск

In [10]:
RUN_TRAIN = CFG.run_train
RUN_EVAL = CFG.run_eval

if RUN_TRAIN:
    train_result = trainer.train()
    trainer.save_model(str(MODEL_OUT / 'final'))
    processor.save_pretrained(MODEL_OUT / 'final')
    with open(MODEL_OUT / 'train_metrics.json', 'w', encoding='utf-8') as f:
        json.dump(train_result.metrics, f, indent=2)
    print('Обучение завершено.')

if RUN_EVAL:
    eval_metrics = trainer.evaluate()
    with open(MODEL_OUT / 'eval_metrics.json', 'w', encoding='utf-8') as f:
        json.dump(eval_metrics, f, indent=2)
    print(eval_metrics)


Step,Training Loss,Validation Loss,Ler,Cer,Ser
4658,1.085300,1.075533,75.794543,74.689506,80.034447
9316,0.977600,0.967082,76.532905,74.126362,79.699371
13974,0.930100,0.926128,75.361156,73.464786,77.810416
18632,0.934200,0.914301,74.847512,72.531049,77.802900


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.
computing LER...: 100%|█████████████████████████████████████████████████████████████| 200/200 [00:01<00:00, 130.25it/s]


Обучение завершено.


computing LER...: 100%|█████████████████████████████████████████████████████████████| 200/200 [00:01<00:00, 129.61it/s]

{'eval_loss': 0.9143317937850952, 'eval_LER': 74.76725521669341, 'eval_CER': 73.20909657760679, 'eval_SER': 78.06908214073216, 'eval_runtime': 1003.8362, 'eval_samples_per_second': 0.199, 'eval_steps_per_second': 0.199, 'epoch': 4.0}


In [11]:
def preview_predictions(num_examples: int = 1):
    subset = dataset['val'].select(range(min(num_examples, len(dataset['val']))))
    outputs = trainer.predict(subset)

    preds = outputs.predictions
    preds = np.where(preds < 0, tokenizer.pad_token_id, preds)

    decoded = processor.batch_decode(preds, skip_special_tokens=False)

    for idx, sample in enumerate(decoded):
        print('=' * 80)
        print(f'Пример {idx}')
        print(sample[:4000])
        print('\nGT:')
        print(subset[idx]['transcription'][:4000])


preview_predictions(5)


computing LER...: 100%|██████████████████████████████████████████████████████████████████| 5/5 [00:01<00:00,  3.17it/s]


Пример 0
<|begin_of_abc|>X:1
T:Title
%%score { ( 1 3 ) | ( 2 4 ) }
L:1/8
Q:1/4=84
M:6/8
I:linebreak $
K:Ab
V:1 treble nm="Grand Piano"
V:3 treble
V:2 bass
V:4 bass
V:1
 [CA][CA][CA] [Ec]>[CA][CA] | [DB]>[CA][CA] [CA]3 | [GB]>[GB][GB] [GB][Gc][Gd] | [GB]3 z3 | %4
w:J.~H.~Grand * * * *||||
 [DB]>[CA][CA] [CA]3 |$ [DGB]>[DGB][DGB] [DGB][Ed][Ec] |[M:5/8] [CEA]2 z z2 |[M:1/8] e |[M:3/4] [Ce]c cd BB | %9
w:|||||
 [A,c]2 A [EB]2 E | [CA]B/c/ d/B/[Ec] AA |$ .[DB]2 G [CA]3 | [CA]>A A[DB] BB | [Ec]c c [Fd]2 [FAd] | %14
w:|||||
 [EAc][Ae] AB dG |[M:3/8] [CEA]2 z |$ [ce]>[df][ce] |[M:3/4] z2 z [Bd]3 | %18
w:||||
 [CE]2 [Bd]- [Bd]/[ce]/[Bd] | z2 z [Ac]3 | z2 z .[Ac]3 | z2 z .[Gd]3 |$ [DGB]2 EA cA | %24
w:|||||
 Bd G [CEA]3 |] %24
w:|
V:2
 A,,A,,A,, A,,>A,,A,, | E,,>A,,A,, A,,3 | [E,E]>[E,E][E,E] [E,E][E,E][E,E] | [E,E

GT:
X:1
T:Title
%%score { ( 1 4 ) | ( 2 3 ) }
L:1/8
Q:1/4=84
M:6/8
I:linebreak $
K:Ab
V:1 treble nm="Grand Piano"
V:4 treble
V:2 bass
V:3 bass
V:1
 [CA][CA][CA] [Ec]>[CA][CA] | [DB]>

In [12]:
def preview_predictions_antirepeat(num_examples: int = 5):
    subset = dataset['val'].select(range(min(num_examples, len(dataset['val']))))

    outputs = trainer.predict(
        subset,
        max_length=CFG.generation_max_length,
        num_beams=1,
        do_sample=False,
        repetition_penalty=1.15,
        no_repeat_ngram_size=4,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    preds = outputs.predictions
    preds = np.where(preds < 0, tokenizer.pad_token_id, preds)
    decoded = processor.batch_decode(preds, skip_special_tokens=False)

    for idx, sample in enumerate(decoded):
        print("=" * 80)
        print(f"Anti-repeat example {idx}")
        print(sample[:4000])
        print("\nGT:")
        print(subset[idx]["transcription"][:4000])


preview_predictions_antirepeat(5)


computing LER...: 100%|██████████████████████████████████████████████████████████████████| 5/5 [00:01<00:00,  3.17it/s]


Anti-repeat example 0
<|begin_of_abc|>X:1
T:Title
%%score { ( 1 3 ) | ( 2 4 ) }
L:1/8
Q:1/4=84
M:6/8
I:linebreak $
K:Ab
V:1 treble nm="Grand Piano"
V:3 treble
V:2 bass
V:4 bass
L:1/4
V:1
 [CA][CA]!~(![CA] [Ec]>[CA]<|end_of_abc|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|